# Check Image quality and shift for all camera

2026.05.01: Health Check before the PFS run

The engineering fibers were used to measure focal plane and check image quality. Because the fiber size is smaller, you cannot use EE5, instead, use EE3.

Central IIS was installed, With it, the following IIS images are used for each camera

* b: Kr 30s and Ne 30s
* r: Ne 30s
* n: Kr 30s
* m: Ne 30s

In [1]:
%load_ext autoreload
%autoreload 2

## Import, Setting

In [2]:
from pfs.lam.opdb import *
from pfs.lam.imqual2Csv import main as getImqual
from pfs.lam.bestFocusPlanefromCsv import main as getBestFocus

from lsst.daf.butler import Butler

In [3]:
from multiprocessing import Pool
import time
import os
import numpy as np
import glob

In [4]:
# To read data manually
from pfs.lam.linePeaksList import filterPeakList
import astropy.io.fits as fio
from pfs.lam.analysisPlot import plotRoiPeak, plotPeaksBrightness
from pfs.lam.detAnalysis import *

### DRP folder

In [10]:
# Hilo
drpPath, repo, rerun = '/work/datastore', 'repo', 'drpActor/reductions'
#drpPath, repo, rerun = '/work/datastore', 'repo', 'PFS/defaults'
datastore = drpPath
collection = [c for c in list(Butler(datastore).registry.queryCollections()) if c.startswith(rerun)]
drpVer='gen3'

### Data processing
Specify `arm`, `specId` and `visit_set_id` (or `experimentId`) for each dataset

#### Common settings to analysis

In [14]:
site = "Subaru"
fiberType = 'ENGINEERING'
outpath = "/work/moritani/spsAIT/202605/throughFocus/"

# Option to find peak
roi_size = 16 #24
seek_size = None
doBck = True

# measurement parameter
if fiberType == 'ENGINEERING':
    criteria = "EE3"
else:
    criteria = "EE5"
piston_index ="motor1"

# Option to define outputs, plots
roiPlot = True
plotPeaksFlux = True
doFit = True    # activate 2D-gaussian fit to measure spot size
doLSF = False
doPrint=False

In [6]:
# specfify the peaklist data
#d_kr = {'b1': '20240419', 'b2': '20240419', 'b3': '20240419', 'b4': '20240419',
#        'n1': '20240822', 'n2': '20250307', 'n3': '20240710', 'n4': '20240724'} 
#d_ne = {'b1': '20240415', 'b2': '20240415', 'b3': '20240415', 'b4': '20240415',
#        'r1': '20240415', 'r2': '20240415', 'r3': '20240415', 'r4': '20240415',
#        'm1': '20251109', 'm2': '20251109', 'm3': '20251109', 'm4': '20251109'}

d_kr = {'b1': '20240419', 'b2': '20240419', 'b3': '20260305', 'b4': '20240419',
#        'n1': '20240822', 'n2': '20250307', 'n3': '20260305', 'n4': '20240724'} 
        'n1': '20240822', 'n2': '20250307', 'n3': '20240710', 'n4': '20240724'} 

d_ne = {'b1': '20240415', 'b2': '20240415', 'b3': '20260305', 'b4': '20240415',
        'r1': '20240415', 'r2': '20240415', 'r3': '20260305', 'r4': '20240415',
        'm1': '20251109', 'm2': '20251109', 'm3': '20251109', 'm4': '20251109'}

**Shift b3/r3/n3 peaklist as it is almost edge recently**

In [ ]:
#df = pd.read_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_kr_n3_20240710_iis.csv")
#df = pd.read_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_kr_b3_20240419_iis.csv")
#df = pd.read_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_ne_b3_20240415_iis.csv")
df = pd.read_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_ne_r3_20240415_iis.csv")

In [ ]:
df['X'] = df.X
df['Y'] = df.Y-5

In [ ]:
#df.to_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_kr_n3_20260305_iis.csv",  index=False)
#df.to_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_kr_b3_20260305_iis.csv", index=False)
#df.to_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_ne_b3_20260305_iis.csv", index=False)
df.to_csv("/work/moritani/spsAIT/202311/peaklist/SM3_peakList_ne_r3_20260305_iis.csv", index=False)

#### Check if processed data processed by drpActor (Hilo) exists

Data on 2026.03.01 (visit 139854..1359859) don't exist

If the data exists, run the cells "By using bulter" If not run the cells "By readind processed image directly"

In [9]:
datastore = '/work/datastore'
collections = ['drpActor/reductions', 'PFS/defaults']

#butler = Butler(datastore, collections=collections, instrument='PFS')
#dataId=dict(visit=140640, spectrograph=4, arm='n')
#butler.get("postISRCCD", dataId)

##### For debug

In [ ]:
dir(postccd)

In [ ]:
%matplotlib inline

In [ ]:
np.nanmin(postccd.image.array), np.nanmax(postccd.image.array)

In [ ]:
fig = 'calexp'; plt.close(fig); fig = plt.figure(fig)

fig, axs = plt.subplots(num=fig, nrows=1, ncols=1, sharex=True, sharey=False,
                         layout='constrained', figsize=(8,8),clear=True)

axs.imshow(postccd.image.array, vmin=0, vmax=100)
plt.show()

##### By reading processed image directly

(2026.05.03) Use the butler to get postOSRCCD

In [15]:
colldir = '20260326T210949Z'

# Select from (n, Kr), (r, Ne), (b, Kr), (b, Ne)

# before the run : test the code
usevisit = {"Ne": 141525, "Kr": 141526}
useseq = {"Ne": 61086, "Kr": 61087}


useexptime = {"Ne": 30, "Kr": 80}

usearm = "n"
useline = "Kr"


experimentId = useseq[useline]  # visit_sequence_id
visitId = usevisit[useline]
arm = usearm
lamps = [useline]
line = useline
exptime = useexptime[useline]


#visitStart, visitEnd = getVisitRange_fromWeb(experimentId, url="http://133.40.164.16/sps-logs/index.html")
#print(f"{experimentId}: {visitStart} -- {visitEnd}")
#visitId = visitStart


#for specId in range(1,5):
for specId in [1]:

    cam = f"{arm}{specId}"
    imname =f'/data/drp/datastore/drpActor/reductions/{colldir}/postISRCCD/*/{visitId}/postISRCCD_PFS_{visitId}_{cam}_drpActor_reductions_{colldir}.fits'
    fname = glob.glob(imname)[0]

    csvPath = os.path.join(outpath,f"sm{specId}",f"Exp{experimentId}",rerun,f"roi{roi_size}",f"doBck{doBck}")

    if not os.path.exists(csvPath):
        os.makedirs(csvPath)


    print(f'Processing {cam}')
    if line == 'Kr':
        peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_kr_{cam}_{d_kr[cam]}_iis.csv"
    elif line == 'Ne':
        peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_ne_{cam}_{d_ne[cam]}_iis.csv"

    peaks = filterPeakList(peaklist, arm, lamps) if peaklist is not None else None

    fits = fio.open(fname)

    dataId=dict(arm=arm, spectrograph=specId, visit=visitId)
    imageInfo = dict(dataId)
    imageInfo.update(filename=fname)
    imageInfo.update(experimentId=experimentId)    

    # Image HDU1
    image = fits[1].data

    if roiPlot and (peaklist is not None):
        RoiPlotTitle = f"Peaklist roiPlot {cam.upper()} Exp{experimentId} - visit{visitId} - roi_size={roi_size}\n"
        plotRoiPeak(image, peaks, roi_size=roi_size, savePlotFile=os.path.join(csvPath,f"{cam}_{visitId}_rawPeak"),raw=True,doSave=True, title=RoiPlotTitle )

    
    data = getFullImageQuality(image, peaklist, imageInfo=imageInfo,
                               roi_size=roi_size, EE=[3,5], seek_size=seek_size,
                               com=True, doBck=doBck, doFit=doFit, doLSF=doLSF, doSep=True,fullSep=False,
                               doPlot=roiPlot, doPrint=doPrint,
                               mask_size=20, threshold=50, subpix=5 , maxPeakDist=80,
                               maxPeakFlux=40000, minPeakFlux=2000, calexpMask=None)

    now = datetime.now() # current date and time\n",
    date_time = now.strftime("%Y%m%dT%Hh%M")
    lwh=None

    csvName = f"Imquality_{cam}_Exp{experimentId}_{visitId}_{date_time}.csv"
    if not os.path.exists(csvPath):
        os.makedirs(csvPath,exist_ok =True)
    data.to_csv(os.path.join(csvPath, csvName))
    if roiPlot:
        RoiPlotName = f"roiPlot_{cam}_Exp{experimentId}_{visitId}_{date_time}"
        RoiPlotTitle = f"roiPlot {cam.upper()} Exp{experimentId} - visitId {visitId} - roi_size={roi_size}\n{date_time}"
        plotRoiPeak(image, data, roi_size, savePlotFile=os.path.join(csvPath, RoiPlotName),raw=False,doSave=True, title=RoiPlotTitle)
    if plotPeaksFlux:
        plotPeaksBrightness(data, doSave=True, savePlotFile=os.path.join(csvPath,f"{cam}_{visitId}_fluxes_{'_'.join(lamps)}{exptime:.0f}s_lwh{lwh}"),
                            plot_title=f"{cam}_{visitId} - {'_'.join(lamps)} exptime {exptime}s lwh {lwh}")

Processing n1
getPeakData FAILED:  not enough values to unpack (expected 1, got 0) cx:4053, cy:3052


##### By using butler

In [19]:
# before the run : test the code
#usevisit = {"Ne": 141442, "Kr": 141451}
#useseq = {"Ne": 60993, "Kr": 61002}

# before the run :
usevisit = {"Ne": 141525, "Kr": 141526}
useseq = {"Ne": 61086, "Kr": 61087}


useexptime = {"Ne": 30, "Kr": 80}

usearm = "m"
useline = "Ne"

# m arm
if usearm=="m":
    usevisit = {"Ne": 141522}
    useseq = {"Ne": 61082}
    

experimentId = useseq[useline]  # visit_sequence_id
visitId = usevisit[useline]
arms = [usearm]
lamps = [useline]
line = useline
exptime = useexptime[useline]



#visitStart, visitEnd = getVisitRange_fromWeb(experimentId, url="http://133.40.164.16/sps-logs/index.html")
#print(f"{experimentId}: {visitStart} -- {visitEnd}")
#visitId = visitStart

for specId in range(1,5):
#for specId in [1]:
    
    for arm in arms:

        cam = f"{arm}{specId}"
        if cam=="n2": continue
        print(f'Processing {cam}')
        if line == 'Kr':
            peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_kr_{cam}_{d_kr[cam]}_iis.csv"
        elif line == 'Ne':
            peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_ne_{cam}_{d_ne[cam]}_iis.csv"
        else:
            print(f"I don't analyze {line} now")
            continue
        getImqual(visitId, peaklist, cam, rerun, experimentId, outpath, drpPath, repo,
                  roi_size, seek_size, doBck, roiPlot, plotPeaksFlux, doFit, doLSF, doPrint, fiberType=fiberType,
                  drpVer=drpVer)

Processing m1
Processing m2
Processing m3
Processing m4
getPeakData FAILED:  not enough values to unpack (expected 1, got 0) cx:2078, cy:12
could not broadcast input array from shape (0,17) into shape (5,17) cx:2078, cy:12


Outputs (example of m arm):

Directory: `outpath`/sm1/Exp56838/drpActor/reductions/roi16/doBckTrue/
    
    ([outpath]/sm[specId]/Exp[experimentId]/drpActor/reductions/roi[roi_size]/doBck[doBck]/)

* m1_136274_rawPeak_roi_all.png : spots centering to position in the peaklist
* m1_136274_fluxes_Ne1s_lwhNone.png  : flux of measured lines
* roiPlot_m1_Exp56838_136274_20260105T13h02_roi_all.png : spots centering to the measured peak position
* Imquality_m1_Exp56838_136274_20260105T13h02.csv measured data

Note that measured spots include false detection (bad pixel etc) too.

Record the file path of csv file to `/work/moritani/spsAIT/camsrecord.yaml`

## plot result

In [20]:
import pandas as pd
from matplotlib import pyplot as plt
import yaml

In [21]:
def plot_spotsize_twopanels(xs, ys, dx, vmin1, vmax1, dy, vmin2, vmax2,
                            xmin=-250, xmax=250, ymin=-250, ymax=250,
                            title='', titlesuf=['x', 'y'], fname='plot', cmap='viridis'):

    # set the font sizes for labels
    plt.rc('xtick', labelsize=10)
    plt.rc('ytick', labelsize=10)

    # scatter plot, with or without ragne limit
    fig = plt.figure(figsize=(8, 3), dpi=90, facecolor='w', edgecolor='k')
    ax1 = fig.add_axes((0.1, 0.15, 0.35, 0.75), aspect='equal')
    ax2 = fig.add_axes((0.6, 0.15, 0.35, 0.75), aspect='equal')

    # x and y axis is the same between the two panels
    ax1.set_xlim(xmin=xmin, xmax=xmax)
    ax1.set_ylim(ymin=ymin, ymax=ymax)
    ax1.set_title(titlesuf[0], fontsize=10)
    ax2.set_xlim(xmin=xmin, xmax=xmax)
    ax2.set_ylim(ymin=ymin, ymax=ymax)
    ax2.set_title(titlesuf[1], fontsize=10)

    sc1 = ax1.scatter(xs, ys, c=dx, vmin=vmin1, vmax=vmax1, marker="o",
                      cmap=cmap, lw=0)
    sc2 = ax2.scatter(xs, ys, c=dy, vmin=vmin2, vmax=vmax2, marker="o",
                      cmap=cmap, lw=0)

    xlname = "X"
    ylname = "Y"

    plt.colorbar(sc1, ax=ax1, label='[pix]')
    plt.colorbar(sc2, ax=ax2, label='[pix]')
    ax1.set_xlabel(xlname, fontsize=10)
    ax1.set_ylabel(ylname, fontsize=10)
    ax2.set_xlabel(xlname, fontsize=10)
    ax2.set_ylabel(ylname, fontsize=10)
    #plt.savefig(fname+".pdf")
    #plt.show()
    fig.suptitle(title, fontsize=10)

    return fig

In [22]:
with open('/work/moritani/spsAIT/camsrecord.yaml', 'rb') as f:
    yml = yaml.safe_load(f)

In [23]:
# dataset  to analyze
date = "202605b"

In [ ]:
yml[date]['b1']['Kr'].split('_')

### b arm

####  b1

In [24]:
cam = 'b1'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile2.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = df1[(df1.brightness<=50000) & (df1.wavelength!=640.40177)] #& (df1.wavelength!=446.49427) & (df1.wavelength!=557.18362) & (df1.wavelength!=587.25432)& (df1.wavelength!=609.78506)   & (df1.wavelength!=638.4756)]  #  
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=540.20631) &(df1.fwhm<10) ]  

In [26]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley],cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.8+/-0.3 pix (x), 1.9+/-1.0 pix (y)


####  b2

In [27]:
cam = 'b2'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = pd.read_csv(iqfile2)
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=640.40177) &(df1.fwhm<10) ]  
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=540.20631) &(df1.fwhm<10) ]  
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=585.41101) & (df1.wavelength!=587.25432) & (df1.wavelength!=594.6481) & (df1.wavelength!=640.40177) & (df1.wavelength!=638.4756)& (df1.wavelength!=609.78506)& (df1.wavelength!=607.60193)]  #  

In [28]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.1+/-1.0 pix (x), 1.9+/-0.7 pix (y)


####  b3

In [29]:
cam = 'b3'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = pd.read_csv(iqfile2)
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=585.41101) & (df1.wavelength!=587.25432) & (df1.wavelength!=594.6481) & (df1.wavelength!=640.40177) & (df1.wavelength!=638.4756)& (df1.wavelength!=609.78506)& (df1.wavelength!=607.60193)]  #  
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=640.40177) &(df1.fwhm<10) ]  
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=540.20631) &(df1.fwhm<10) ]  

In [30]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 =plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.8+/-0.5 pix (x), 2.0+/-1.3 pix (y)


####  b4

In [31]:
cam = 'b4'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile1.split('_')[-2]

df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = pd.read_csv(iqfile2)
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=585.41101) & (df1.wavelength!=587.25432) & (df1.wavelength!=594.6481) & (df1.wavelength!=640.40177) & (df1.wavelength!=638.4756)& (df1.wavelength!=609.78506)& (df1.wavelength!=607.60193)]  #  
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=640.40177) &(df1.fwhm<10) ]  
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=540.20631) &(df1.fwhm<10) ]  

In [32]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.7+/-0.3 pix (x), 2.0+/-1.0 pix (y)


### r arm

####  r1

In [35]:
cam = 'r1'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000) &(df1.fwhm<10)]
if date == '202603l':
    # affected by CR
    df1=df1[~((df1.fiber==471)&(df1.wavelength==892.19496))]

In [36]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.0+/-0.7 pix (x), 2.0+/-0.2 pix (y)


####  r2

In [39]:
cam = 'r2'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) &(df1.fwhm<10)]

In [40]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.9+/-0.9 pix (x), 2.1+/-0.9 pix (y)


####  r3

In [42]:
cam = 'r3'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) &(df1.fwhm<10)]

In [43]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.2+/-1.7 pix (x), 2.0+/-0.2 pix (y)


####  r4

In [44]:
cam = 'r4'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) &(df1.fwhm<10)]

In [46]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.9+/-0.1 pix (x), 2.2+/-1.0 pix (y)


### n arm

####  n1

In [47]:
cam = 'n1'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=55000) & (df1.fwhm<=10) & (df1.wavelength!=1182.26136) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [48]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.2+/-1.8 pix (x), 2.8+/-2.8 pix (y)


####  n2 --- missing

In [ ]:
cam = 'n2'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=60000) & (df1.fwhm<=10) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [ ]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
#print(f'x:{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f}, y:{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f}')

fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')

####  n3

In [49]:
cam = 'n3'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=55000) & (df1.fwhm<=10) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [50]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.1+/-1.5 pix (x), 2.6+/-2.3 pix (y)


####  n4

In [51]:
cam = 'n4'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=55000) & (df1.fwhm<=10) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [52]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.6+/-2.6 pix (x), 3.0+/-2.4 pix (y)


### m arm

####  m1

In [57]:
cam = 'm1'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000) & (df1.fwhm<=10)]

In [56]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.9+/-0.1 pix (x), 2.2+/-1.1 pix (y)


####  m2

In [58]:
cam = 'm2'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000) & (df1.fwhm<=10)]

In [59]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 2.0+/-0.9 pix (x), 2.0+/-0.3 pix (y)


####  m3

In [60]:
cam = 'm3'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000) & (df1.fwhm<=10)]

In [61]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.9+/-0.2 pix (x), 2.0+/-0.6 pix (y)


####  m4

In [62]:
cam = 'm4'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000) & (df1.fwhm<=10)]

In [63]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')
print(f"Mean: {titlex} (x), {titley} (y)")

Mean: 1.9+/-0.6 pix (x), 2.3+/-1.2 pix (y)


## Spot shift

In [64]:
def plot_spotshift_twopanels(df, xrange=2., yrange=2., detx=4176, dety=4176, title=''):


    fig = "comp_pos"; plt.close(fig);
    fig, axs = plt.subplots(num=fig, nrows=1, ncols=2, sharex=False, sharey=False,
                            figsize=(8,3))

    # scatter  
    axs[0].scatter(df.oid_x_x-df.oid_x_y, df.oid_y_x-df.oid_y_y)
    dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
    dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
    axs[0].scatter(dx_mean, dy_mean)

    axs[0].set_xlim(xmin=-1*xrange, xmax=xrange)
    axs[0].set_ylim(ymin=-1*yrange, ymax=yrange)
    axs[0].set_aspect('equal')

    # quiver
    axs[1].quiver(df.oid_x_y, df.oid_y_y, df.oid_x_x-df.oid_x_y, df.oid_y_x-df.oid_y_y, color='dimgrey', scale=1e+2,label='before')

    axs[1].set_xlim(xmin=0., xmax=detx)
    axs[1].set_ylim(ymin=0., ymax=dety)
    axs[1].set_aspect('equal')

        
    fig.suptitle(title, fontsize=9)

    #plt.show()

    return fig

In [65]:
# Previous measurement
date_old='202603b'

### b arm

#### b1

In [66]:
cam = 'b1'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [67]:
# plot
xrange=6.
yrange=6.
detsize=4176
dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)
fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346),Kr(140345), New: Ne(141525),Kr(141526) b1
median:(dx,dy)=(-0.0, -0.3) pix


#### b2

In [68]:
cam = 'b2'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [69]:
# plot
xrange=5.
yrange=5.
detsize=4176
dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346),Kr(140345), New: Ne(141525),Kr(141526) b2
median:(dx,dy)=(0.5, 3.3) pix


#### b3

In [70]:
cam = 'b3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [71]:
# plot
xrange=5.
yrange=5.
detsize=4176
dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346),Kr(140345), New: Ne(141525),Kr(141526) b3
median:(dx,dy)=(-1.3, 8.1) pix


#### b4

In [72]:
cam = 'b4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [73]:
# plot
xrange=5.
yrange=5.
detsize=4176
dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346),Kr(140345), New: Ne(141525),Kr(141526) b4
median:(dx,dy)=(-0.2, 0.4) pix


### r arm

#### r1

In [74]:
cam = 'r1'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [75]:
# plot
xrange=6.
yrange=6.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346), New: Ne(141525) r1
median:(dx,dy)=(-0.1, -0.3) pix


#### r2

In [76]:
cam = 'r2'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [77]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346), New: Ne(141525) r2
median:(dx,dy)=(0.4, 3.1) pix


#### r3

In [78]:
cam = 'r3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [79]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346), New: Ne(141525) r3
median:(dx,dy)=(-1.2, 8.0) pix


#### r4

In [80]:
cam = 'r4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [81]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Ne(140346), New: Ne(141525) r4
median:(dx,dy)=(-0.3, 0.4) pix


### n arm

#### n1

In [82]:
cam = 'n1'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [83]:
# plot
xrange=6.
yrange=6.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Kr(140345), New: Kr(141526) n1
median:(dx,dy)=(0.0, 0.2) pix


#### n2

In [ ]:
cam = 'n2'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [ ]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### n3

In [84]:
cam = 'n3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [85]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Kr(140345), New: Kr(141526) n3
median:(dx,dy)=(-0.8, -7.0) pix


#### n4

In [86]:
cam = 'n4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [87]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')
print(title)

Old: Kr(140345), New: Kr(141526) n4
median:(dx,dy)=(-0.4, -0.4) pix


### m arm 

#### m1

In [ ]:
cam = 'm1'
# Old
# Ne
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Ne
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [ ]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### m2

In [ ]:
cam = 'm2'
# Old
# Ne
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Ne
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [ ]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### m3

In [ ]:
cam = 'm3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [ ]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### m4

In [ ]:
cam = 'm4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [ ]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')